# Candidate new bicycle HIN corridors

This notebook screens 2024–Sept. 1, 2026 CRIS crashes outside the City's official 22 Bicycle HIN corridors. It creates variable-length candidate stretches by road, using the longest official corridor as a maximum stretch length. This is an exploratory approximation, not the City's unpublished scoring model.

In [1]:
from pathlib import Path
import zipfile
import requests
import pandas as pd
import geopandas as gpd
from sklearn.cluster import AgglomerativeClustering

ROOT = Path.cwd()
RAW = ROOT / 'data' / 'raw'
BOUNDARIES = ROOT / 'data' / 'boundaries'
OUT = ROOT / 'outputs'
BOUNDARIES.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)

/Users/eshaan.sarup/Library/CloudStorage/OneDrive-Hearst/Documents/GitHub/san-antonio-bicyclist-crashes/.venv-2/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Load current qualifying bicyclist crashes.
raw = pd.read_csv(RAW / 'myexport_final.csv', skiprows=12, low_memory=False)
severity = {'K - FATAL INJURY', 'A - SUSPECTED SERIOUS INJURY'}
target = raw[(raw['Person Type'] == '3 - PEDALCYCLIST') & raw['Person Injury Severity'].isin(severity)].copy()
target['year'] = pd.to_numeric(target['Crash Year'], errors='coerce')
target['latitude'] = pd.to_numeric(target['Latitude'], errors='coerce')
target['longitude'] = pd.to_numeric(target['Longitude'], errors='coerce')
target['death'] = (target['Person Injury Severity'] == 'K - FATAL INJURY').astype(int)
target['serious_injury'] = (target['Person Injury Severity'] == 'A - SUSPECTED SERIOUS INJURY').astype(int)
sa = target[(target['City'] == 'SAN ANTONIO') & target['year'].between(2024, 2026)].copy()
crashes = (sa.groupby('Crash ID', as_index=False).agg(year=('year','first'), latitude=('latitude','first'), longitude=('longitude','first'), deaths=('death','sum'), serious_injuries=('serious_injury','sum')))
geo = crashes.dropna(subset=['latitude','longitude'])
points = gpd.GeoDataFrame(geo, geometry=gpd.points_from_xy(geo['longitude'], geo['latitude']), crs=4326)

# Load official corridors and identify current crashes already inside them.
hin_url = ('https://services.arcgis.com/g1fRTDLeMgspWrYp/arcgis/rest/services/'
           'SS4A_HIN_Dashboard_Data/FeatureServer/1/query?where=1%3D1&outFields=*'
           '&returnGeometry=true&outSR=4326&f=geojson')
response = requests.get(hin_url, timeout=120)
response.raise_for_status()
hin_file = BOUNDARIES / 'bicycle_hin_corridors.geojson'
hin_file.write_bytes(response.content)
hin = gpd.read_file(hin_file).to_crs(4326).rename(columns={'Name':'corridor'})
hin_buffer = hin.to_crs(2278).copy()
hin_buffer['geometry'] = hin_buffer.geometry.buffer(150)
official_matches = gpd.sjoin(points.to_crs(2278), hin_buffer[['bicycle_hin_id','geometry']], how='inner', predicate='within')
official_ids = set(official_matches['Crash ID'])
outside_points = points[~points['Crash ID'].isin(official_ids)].copy()
print('Current qualifying crashes:', len(points))
print('Current crashes inside official HIN:', len(official_ids))
print('Current crashes outside official HIN:', len(outside_points))

Current qualifying crashes: 80
Current crashes inside official HIN: 3
Current crashes outside official HIN: 77


In [3]:
# Match outside-HIN crashes to streets and build variable-length candidates.
streets_dir = RAW / 'streets'
street_shp = streets_dir / 'Streets' / 'Streets.shp'
if not street_shp.exists():
    with zipfile.ZipFile(RAW / 'Streets.zip') as archive:
        archive.extractall(streets_dir)
roads = gpd.read_file(street_shp)
roads = roads.rename(columns={'CartID':'segmentid','MSAG_NAME':'road_label','FROM_STREE':'from_street','TO_STREET':'to_street','CoSARoadFu':'road_class'})
roads['length_miles'] = roads['LengthFeet'] / 5280
roads = roads[roads['length_miles'] > 0].copy()
matches = gpd.sjoin_nearest(outside_points.to_crs(roads.crs), roads[['segmentid','road_label','from_street','to_street','road_class','length_miles','geometry']], how='left', distance_col='match_distance_ft')
matches = matches[matches['match_distance_ft'] <= 150].sort_values('match_distance_ft').drop_duplicates('Crash ID').copy()

# The longest official corridor is the maximum candidate stretch.
max_official_miles = float(hin['Miles'].max())
matches['candidate_group'] = pd.NA
group_number = 0
for road_name, road_group in matches.groupby('road_label', dropna=False):
    if len(road_group) < 2:
        continue
    xy = road_group[['geometry']].copy()
    coordinates = [[point.x, point.y] for point in xy.geometry]
    model = AgglomerativeClustering(n_clusters=None, distance_threshold=max_official_miles * 5280, linkage='complete')
    labels = model.fit_predict(coordinates)
    for local_label in sorted(set(labels)):
        indexes = road_group.index[labels == local_label]
        if len(indexes) >= 2:
            matches.loc[indexes, 'candidate_group'] = group_number
            group_number += 1
matches = matches[matches['candidate_group'].notna()].copy()

def join_values(series):
    values = sorted({str(value).strip() for value in series.dropna() if str(value).strip() and str(value).strip().upper() != 'TBD'})
    return ', '.join(values)
candidates = (matches.groupby('candidate_group', as_index=False)
    .agg(crashes=('Crash ID','nunique'), deaths=('deaths','sum'), serious_injuries=('serious_injuries','sum'), first_year=('year','min'), last_year=('year','max'), roads=('road_label',join_values), from_streets=('from_street',join_values), to_streets=('to_street',join_values), max_match_distance_ft=('match_distance_ft','max')))
spans = []
for group_id, geometry in matches.groupby('candidate_group')['geometry']:
    x_values = [point.x for point in geometry]
    y_values = [point.y for point in geometry]
    spans.append({'candidate_group': group_id, 'span_miles': max(max(x_values)-min(x_values), max(y_values)-min(y_values)) / 5280})
candidates = candidates.merge(pd.DataFrame(spans), on='candidate_group', how='left')
candidates = candidates.sort_values(['crashes','deaths','serious_injuries'], ascending=False)
candidates.to_csv(OUT / 'candidate_new_hin_corridors_2024_2026.csv', index=False)
print('Candidate new corridors outside official HIN:', len(candidates))
display(candidates)

Candidate new corridors outside official HIN: 9


,candidate_group,crashes,deaths,serious_injuries,first_year,last_year,roads,from_streets,to_streets,max_match_distance_ft,span_miles
3,3,3,2,1,2024,2025,HISTORIC OLD HWY 90,"MONTEREY ST, S SAN JOAQUIN, W CESAR CHAVEZ BLVD","MONTEREY ST, SW 36TH ST",1.665400,0.540428
2,2,2,1,1,2024,2025,EVERS RD,"NW LOOP 410, WURZBACH RD","JOINER DR, NW LOOP 410 ACCESS RD",3.044905,0.187049
8,8,2,1,1,2025,2025,WALZEM RD,,,1.417411,1.247692
0,0,2,0,2,2024,2026,AUSTIN HWY,WALZEM RD,NE LOOP 410 ACCESS RD,8.567862,1.003890
1,1,2,0,2,2024,2024,E HOUSTON ST,"N ST MARYS, SOLEDAD ST","BRIDGE, NAVARRO ST",4.589853,0.205569
4,4,2,0,2,2024,2026,IH 35 N ACCESS RD,,,33.498470,0.283325
5,5,2,0,2,2024,2026,NOGALITOS ST,"PRADO ST, PRUITT","CONCEPTION CT, DRAKE AVE",4.616533,0.693866
6,6,2,0,2,2024,2025,ROOSEVELT AVE,HERBST,SE LOOP 410 ACCESS RD,9.087353,0.834120
7,7,2,0,2,2025,2026,S FLORES ST,"E HARDING BLVD, KENDALIA AVE","E BONNER AVE, GENEVIEVE DR",4.328716,0.640965


In [4]:
# Interactive map of the candidate corridors and every current crash.
import html
import folium

# Build corridor lines from the actual mapped street segments used by each candidate.
segment_groups = matches[['segmentid','candidate_group']].dropna().drop_duplicates()
candidate_roads = roads.merge(segment_groups, on='segmentid', how='inner')
candidate_lines = candidate_roads.dissolve(by='candidate_group', as_index=False)
candidate_lines = candidate_lines.merge(candidate_corridors.drop(columns=['roads','from_streets','to_streets','length_miles']), on='candidate_group', how='left')

# Add the nearest road name to every current crash for the hover popup.
all_current_matches = gpd.sjoin_nearest(current_points.to_crs(roads.crs), roads[['segmentid','road_label','geometry']], how='left', distance_col='match_distance_ft')
all_current_matches = all_current_matches.sort_values('match_distance_ft').drop_duplicates('Crash ID')
point_info = points.merge(all_current_matches[['Crash ID','road_label']], on='Crash ID', how='left')
point_info = point_info[point_info['year'].between(2024, 2026)].copy()

m = folium.Map(location=[point_info.geometry.y.mean(), point_info.geometry.x.mean()], zoom_start=11, tiles='OpenStreetMap', control_scale=True)
folium.GeoJson(candidate_lines.to_crs(4326).to_json(), name='Candidate corridors', style_function=lambda feature: {'color':'#d95f02','weight':6,'opacity':0.85}, tooltip=folium.GeoJsonTooltip(fields=['roads','crashes','deaths','serious_injuries','first_year','last_year'], aliases=['Road','Crashes','Deaths','Serious injuries','First year','Last year'], sticky=False)).add_to(m)

for _, row in point_info.iterrows():
    outcome = f"{int(row['deaths'])} death(s), {int(row['serious_injuries'])} serious injury/ies"
    popup = (f"<b>Crash {html.escape(str(row['Crash ID']))}</b><br>"
             f"Year: {int(row['year'])}<br>Road: {html.escape(str(row.get('road_label', '')))}<br>"
             f"Outcome: {outcome}<br>Latitude: {row['latitude']:.6f}<br>Longitude: {row['longitude']:.6f}")
    folium.CircleMarker(location=[row['latitude'], row['longitude']], radius=5, color='#e67e22' if row['deaths'] else '#3478a4', fill=True, fill_opacity=0.9, tooltip=f"{int(row['year'])} — Crash {row['Crash ID']}", popup=folium.Popup(popup, max_width=300)).add_to(m)

folium.LayerControl().add_to(m)
map_path = OUT / 'candidate_corridors_map_2024_2026.html'
m.save(map_path)
print('Map saved to:', map_path)

NameError: name 'candidate_corridors' is not defined

## Interpretation

The candidate table and map are exploratory. A candidate requires at least two current-period qualifying crashes on the same named roadway within a bounded stretch no longer than the City's longest official corridor. The map shows our candidate stretches in orange and individual crashes as clickable points.